# 02. 결과 분석

평가항목 2(모델 비교와 선정근거), 3(영향요인·오류분석), 4(현장 활용)의 근거를 정리하는 노트북입니다.

- **먼저 프로젝트 폴더에서 `python run.py`를 실행**해 `outputs/`를 만들어야 합니다. 이 노트북은 그 결과를 읽어 분석합니다.
- 6절에서는 최종 모델을 `src`의 함수로 다시 학습해 영향변수를 분석하고, `run.py`의 예측과 같은지 확인합니다.
- 수정한 뒤에는 항상 **커널 재시작 → 전체 실행**이 통과하는지 확인하세요.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.experiment import wavg, error_slices
from src.metrics import peak_events
from src.viz import setup_plot_style, heatmap, DOW_KR

setup_plot_style()
pd.set_option("display.max_columns", 50)
DATA, OUT = ROOT / C.DATA_PATH, ROOT / C.OUTPUT_DIR

required = ["metrics_cv.csv", "metrics_test.csv", "predictions_test.csv", "run_info.json"]
missing = [f for f in required if not (OUT / f).exists()]
if missing:
    raise FileNotFoundError(f"먼저 프로젝트 폴더에서 `python run.py`를 실행하세요. 없는 파일: {missing}")

cv = pd.read_csv(OUT / "metrics_cv.csv")
test = pd.read_csv(OUT / "metrics_test.csv")
preds = pd.read_csv(OUT / "predictions_test.csv", parse_dates=["ts", "date"])
info = json.loads((OUT / "run_info.json").read_text(encoding="utf-8"))
selected = info["selected_model"]
thr = float(test["peak_thr"].iloc[0])
RISK = f"gbm_q{int(C.RISK_QUANTILE * 100)}"
print("GBM 백엔드:", info["gbm_backend"])
print("선정 모델:", selected, "/ 피크 임계값:", round(thr, 1), "kW")

## 1. 교차검증 비교 (모델 선정 근거)

7/5부터 주 단위 walk-forward 7개 폴드의 결과입니다. 폴드별 지표를 채점일 수로 가중평균했습니다. 선정 규칙(점예측 모델 중 CV MAE 최소)은 실험 전에 고정했습니다.

In [ ]:
summary = pd.DataFrame([
    {"모델": m, "종류": g["kind"].iloc[0], "MAE": wavg(g, "mae"), "일최대 오차": wavg(g, "daily_max_mae"),
     "피크 Recall": wavg(g, "recall"), "피크 F1": wavg(g, "f1"), "폴드 간 MAE 표준편차": g["mae"].std()}
    for m, g in cv.groupby("model")
]).sort_values("MAE")
display(summary.round(3))

fold_mae = cv.pivot(index="fold", columns="model", values="mae")
ax = fold_mae.plot(marker="o", figsize=(10, 4))
ax.set_ylabel("MAE (kW)")
ax.set_xlabel("검증 주 시작일")
ax.set_title("폴드별 MAE: 휴가 주간(08-02, 08-09)에서 나이브가 무너짐")
ax.legend(fontsize=8)
plt.show()

정상 주간에는 "지난주 같은 시각" 나이브도 강하지만, 휴가처럼 불규칙한 주간에서 크게 무너집니다. 모델의 가치는 이런 주간을 달력·생산계획 피처로 버텨내는 데서 나옵니다.

## 2. 최종 테스트 (8/23~9/14, 선정에 사용하지 않은 기간)

In [ ]:
cols_show = ["model", "kind", "mae", "rmse", "nmae", "hourly_mae", "daily_max_mae", "daily_max_bias",
             "peak_time_hit_1h", "precision", "recall", "f1", "fn", "fp"]
display(test[cols_show].round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
t = test.set_index("model")
t["mae"].sort_values().plot(kind="barh", ax=axes[0], color="#8aa")
axes[0].set_xlabel("MAE (kW)")
axes[0].set_title("테스트 MAE")
t[["fn", "fp"]].plot(kind="barh", ax=axes[1])
axes[1].set_xlabel("건수 (15분 슬롯)")
axes[1].set_title("놓친 피크(FN)와 헛경보(FP)")
plt.tight_layout()
plt.show()

테스트 구간은 정상 주간뿐이라 CV보다 오차가 훨씬 작습니다. 불규칙 주간의 성능은 1절의 CV로, 정상 운영 성능은 이 절의 테스트로 설명하세요. 분위수 모델(`gbm_q80`)은 점예측 오차는 크지만 놓친 피크(FN)가 가장 적어, 피크 경보용으로 역할이 다릅니다.

## 3. 검증 설계 근거: 누수 시연과 ablation

`python run.py --fast`로 실행했다면 이 절의 파일이 없어 건너뜁니다.

In [ ]:
for name, title in [("leakage_demo.csv", "분할 방식에 따른 점수 부풀림 (GBM)"), ("ablation_cv.csv", "설정별 비교 (GBM, CV 기준)")]:
    path = OUT / name
    if path.exists():
        print(title)
        display(pd.read_csv(path).round(3))
    else:
        print(f"{name} 없음: `python run.py`(전체 실행)로 생성하세요.")

## 4. 테스트 예측 살펴보기

In [ ]:
w = preds[preds["ts"] < preds["ts"].min().normalize() + pd.Timedelta(days=14)]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(w["ts"], w["kw"], color="black", lw=1, label="실측")
ax.plot(w["ts"], w[selected], lw=1, label=selected)
ax.plot(w["ts"], w["naive_lastweek"], lw=0.8, alpha=0.6, label="naive_lastweek")
ax.axhline(thr, ls="--", lw=0.8, color="red", label=f"피크 임계값 {thr:.0f} kW")
ax.set_ylabel("15분 수요 (kW)")
ax.set_title("테스트 첫 2주: 실측과 예측")
ax.legend(ncol=4, fontsize=8, loc="upper right")
plt.show()

daily = preds.groupby("date")[["kw", selected, RISK]].max()
fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={"width_ratios": [2, 1]})
axes[0].plot(daily.index, daily["kw"], "o-", color="black", ms=3, label="실측 일 최대")
axes[0].plot(daily.index, daily[selected], "o-", ms=3, label=f"{selected} 일 최대")
axes[0].plot(daily.index, daily[RISK], "o-", ms=3, alpha=0.7, label=f"{RISK} 일 최대")
axes[0].axhline(thr, ls="--", lw=0.8, color="red")
axes[0].set_ylabel("일 최대 15분 수요 (kW)")
axes[0].legend(fontsize=8)
axes[0].set_title("일 최대수요: 점예측은 체계적으로 낮게 잡음")
axes[1].scatter(preds["kw"], preds[selected], s=3, alpha=0.3)
lim = [0, preds["kw"].max() * 1.05]
axes[1].plot(lim, lim, "r--", lw=1)
axes[1].set_xlabel("실측 (kW)")
axes[1].set_ylabel("예측 (kW)")
axes[1].set_title(f"{selected}: 실측 vs 예측")
plt.tight_layout()
plt.show()

트리 모델은 잎에 모인 학습 타깃의 평균을 예측하므로 극단값이 깎입니다. 일 최대수요를 낮게 잡는 경향(`daily_max_bias` 음수)이 그 결과이고, 피크 경보에는 분위수 모델을 따로 쓰는 이유입니다.

## 5. 오류 분석: 오차와 FN·FP가 몰리는 조건 (평가항목 3)

In [ ]:
sl = error_slices(preds, selected, thr)
h = sl["by_hour"]
fig, ax1 = plt.subplots(figsize=(11, 4))
ax1.bar(h["hour"], h["mae"], color="#8aa")
ax1.set_xlabel("시간")
ax1.set_ylabel("MAE (kW)")
ax2 = ax1.twinx()
ax2.plot(h["hour"], h["fn"], "o-", color="red", label="FN (놓친 피크)")
ax2.plot(h["hour"], h["fp"], "s--", color="orange", label="FP (헛경보)")
ax2.set_ylabel("건수")
ax2.legend(loc="upper left", fontsize=8)
ax1.set_title("시간대별 오차와 FN·FP")
plt.show()

display(sl["by_dow"].assign(요일=lambda d: d["dow"].map(DOW_KR)).round(2))
display(sl["by_prod"].round(2))

In [ ]:
d = preds.assign(
    fn=(preds["kw"] >= thr) & (preds[selected] < thr),
    fp=(preds["kw"] < thr) & (preds[selected] >= thr),
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col, title in [(axes[0], "fn", "FN (놓친 피크) 건수"), (axes[1], "fp", "FP (헛경보) 건수")]:
    m = d[d["dow"] <= 5].pivot_table(index="dow", columns="hour", values=col, aggfunc="sum").rename(index=DOW_KR)
    m = m.loc[:, (m.sum() > 0)]
    heatmap(m, ax=ax, cmap="Reds", annotate=True, cbar_label="건수")
    ax.set_xlabel("시간")
    ax.set_title(f"요일 × 시간 {title} (평일)")
plt.tight_layout()
plt.show()

top_fn = (d[d["fn"]].groupby(["dow", "hour"]).size().sort_values(ascending=False).head(8)
          .rename("FN 건수").reset_index().assign(요일=lambda x: x["dow"].map(DOW_KR)))
display(top_fn)

보고서에는 FN·FP가 몰리는 조건을 공정 언어로 옮겨 쓰세요. 예를 들어 "월요일 오전(재가동 직후)", "오후 13~16시", "생산량 501~1500 구간"처럼요. 요일별 편향(`bias`)이 음수면 그 요일을 낮게 예측한다는 뜻입니다.

## 6. 영향 변수와 상호작용 (평가항목 3)

최종 모델을 `src`의 함수로 다시 학습합니다. 시드와 설정이 같으므로 `run.py`의 예측과 일치해야 합니다(재현성 확인).

In [ ]:
from src.data import load_raw, clean
from src.features import to_long, build_features, feature_columns
from src.experiment import train_test, fit_predict
from src.models import make_models

hourly, _, _ = clean(load_raw(DATA))
feat = build_features(to_long(hourly))
cols = feature_columns()
tr, te = train_test(feat)
_, model = make_models()[selected]
p, _ = fit_predict(model, tr, te, cols)
print("run.py 예측과의 최대 차이:", float(np.abs(p - preds[selected].to_numpy()).max()))

**순열 중요도**: 피처 하나를 무작위로 섞었을 때 테스트 MAE가 얼마나 나빠지는지로 중요도를 잽니다. 서로 비슷한 정보를 가진 피처(예: 래그들, 생산량 피처들)는 하나를 섞어도 다른 피처가 대신하므로 중요도가 낮게 나올 수 있습니다.

In [ ]:
from sklearn.inspection import permutation_importance, PartialDependenceDisplay

pi = permutation_importance(model, te[cols], te["kw"], scoring="neg_mean_absolute_error",
                            n_repeats=5, random_state=C.SEED)
imp = pd.Series(pi.importances_mean, index=cols).sort_values()
ax = imp.plot(kind="barh", figsize=(7, 8), color="#8aa")
ax.set_xlabel("섞었을 때 MAE 증가 (kW)")
ax.set_title("순열 중요도 (테스트 구간)")
plt.show()
display(imp.sort_values(ascending=False).head(10).round(2).rename("MAE 증가 (kW)"))

**부분 의존도(PDP)**: 다른 피처는 그대로 두고 한 피처 값만 바꿨을 때 평균 예측이 어떻게 변하는지 봅니다. 2차원 PDP로는 두 피처의 상호작용(한 피처의 효과가 다른 피처 값에 따라 달라지는지)을 봅니다.

In [ ]:
# 최신 scikit-learn은 정수형 피처의 PDP를 막으므로(값이 반올림될 수 있어서) 실수형으로 바꿔 넣는다.
# 트리 모델은 값의 순서만 보므로 예측 결과는 달라지지 않는다.
X = te[cols].sample(min(1000, len(te)), random_state=C.SEED).astype(float)
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
PartialDependenceDisplay.from_estimator(model, X, ["hour", "temp", "prod_h"], ax=axes)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
PartialDependenceDisplay.from_estimator(model, X, [("hour", "dow"), ("hour", "prod_h")], ax=axes)
plt.tight_layout()
plt.show()

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(model)
    sample = te[cols].sample(min(2000, len(te)), random_state=C.SEED)
    shap.summary_plot(explainer.shap_values(sample), sample, show=False)
    plt.show()
except ImportError:
    print("shap 미설치: `pip install shap` 후 다시 실행하면 SHAP 요약 그림이 추가됩니다.")
except Exception as e:
    print("SHAP 계산을 건너뜁니다:", e)

## 7. 익일 피크 위험 캘린더 (평가항목 4 초안)

분위수 모델(`gbm_q80`)의 시간별 최대 예측으로 "어느 날 몇 시에 피크 위험이 있는가"를 보여줍니다. × 표시는 실제로 피크가 발생한 시간입니다. 운영에서는 매일 자정에 다음 날 캘린더를 만들어 설비 기동 순서나 작업 배치를 조정하는 데 씁니다.

In [ ]:
hr = preds.groupby(["date", "hour"])[[RISK, selected, "kw"]].max().reset_index()
cal = hr.pivot(index="date", columns="hour", values=RISK)
act = hr.pivot(index="date", columns="hour", values="kw")
labels = [f"{d:%m-%d} ({DOW_KR[d.dayofweek + 1]})" for d in cal.index]

fig, ax = plt.subplots(figsize=(13, 8))
heatmap(cal.set_axis(labels, axis=0), ax=ax, cmap="Reds", cbar_label=f"{RISK} 시간별 최대 예측 (kW)")
from matplotlib.patches import Rectangle
ry, rx = np.where(cal.to_numpy() >= thr)
for y, x in zip(ry, rx):
    ax.add_patch(Rectangle((x - 0.5, y - 0.5), 1, 1, fill=False, edgecolor="tab:blue", lw=1.3))
yy, xx = np.where(act.to_numpy() >= thr)
ax.scatter(xx, yy, marker="x", color="black", s=18, label="실제 피크 발생")
ax.add_patch(Rectangle((0, 0), 0, 0, fill=False, edgecolor="tab:blue", lw=1.3, label="위험 예측 (임계값 이상)"))
ax.set_xlabel("시간")
ax.set_title(f"피크 위험 캘린더 (임계값 {thr:.0f} kW): 파란 칸 = 경보, × = 실제 피크")
ax.legend(loc="upper right", fontsize=8)
plt.show()

hourly_eval = pd.DataFrame([
    {"모델": m, **peak_events(hr["kw"], hr[m], thr)} for m in [selected, RISK]
]).drop(columns=["peak_thr"])
print("시간 단위 피크 경보 성능 (한 시간 안에 15분 피크가 하나라도 있으면 양성):")
display(hourly_eval.round(3))

캘린더에서 **경보가 하루 통째로 빠진 날**이 있다면 전날이 정전·휴무였는지 확인하세요. 래그 피처(`lag_1d`, `prev_day_*`)가 비정상적으로 낮아져 모델이 "조용한 날"로 착각하는 경우입니다.

## 8. 다음 작업

- **피크저감 시뮬레이션 (평가항목 4):** 08시 순차기동, 주간 내 부하 재배치, 야간 이동 시나리오별로 "피크 kW 감소 × 기본요금 − 인건비 증가"를 비교합니다. 인건비 가중치는 `hourly["labor_mult"]`에 있습니다.
- **경보 임계값 결정:** FN과 FP의 비용 비율을 정하고, 분위수(`RISK_QUANTILE`)를 그 기준으로 고릅니다.
- **모델 개선:** LightGBM 튜닝, GRU 비교 모델, 월요일·토요일 전환 피처 보강 후 이 노트북을 다시 실행해 변화를 확인합니다.